In [9]:
import pandas as pd
import numpy as np
from EssSimulation_withoutMaxDemand import EssSimulationModel
import calendar
import copy
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif']=['SimHei']    # 用来正常显示中文标签
plt.rcParams['axes.unicode_minus'] = False    # 用来显示负号

In [10]:
exp_name = "estimate1016"
node_name = "route_B"

In [11]:
es_info = {"transform_capacity": 630000,
           "invertband": 0,
           "soc_redundant_ratio": 0,
            "usable_depth": 0.95,
            "charge_loss": 0.92,
            "discharge_loss": 0.95,
            "es_charge_max": 12500,
            "es_charge_min": -12500,
            "es_capacity_max": 25000,
            "es_capacity_min": 0}
max_demand_price = 37

In [12]:
def get_monthly_max_load(df: pd.DataFrame):
    """
    从一个以时间索引的 DataFrame 中，提取每个月 'load' 列的最大值。

    参数:
        df (pd.DataFrame): 输入的 DataFrame，其 index 必须是时间对象 (DatetimeIndex)。

    返回:
        List[float]: 一个包含每个月 'load' 列最大值的列表，按时间顺序排列。

    异常:
        KeyError: 如果 DataFrame 中不存在 'load' 列。
        TypeError: 如果 DataFrame 的 index 不是 DatetimeIndex。
    """
    # 检查 'load' 列是否存在
    if 'total_load' not in df.columns:
        raise KeyError("DataFrame must have a 'load' column.")

    # 检查 index 是否为 DatetimeIndex
    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError("DataFrame index must be a DatetimeIndex.")

    # 使用 resample 方法按月分组，并获取每个月 'load' 列的最大值
    # 'M' 表示按月的末尾进行分组
    monthly_total_load_max = df['total_load'].resample('M').max()
    monthly_demand_load_max = df['demand_load'].resample('M').max()
    monthly_diff = monthly_total_load_max - monthly_demand_load_max

    # 将结果转换为列表并返回
    return monthly_total_load_max.tolist(), monthly_demand_load_max.tolist()

In [13]:
demand_load_df = pd.read_csv(f"./data/{exp_name}/{node_name}/demand_load.csv")
demand_load_df['time'] = pd.to_datetime(demand_load_df['time'])
demand_load_df.set_index('time', inplace=True)

strategy_df = pd.read_csv(f"./data/{exp_name}/{node_name}/opt_result/schedule_result_evencharge_new_dod95.csv")
strategy_df.rename(columns={"power_opt": "value"}, inplace=True)
strategy_df['time'] = pd.to_datetime(strategy_df['time'])
strategy_df.set_index('time', inplace=True)

ele_price_df = pd.read_csv(f"./data/{exp_name}/{node_name}/ele_price.csv")
ele_price_df['time'] = pd.to_datetime(ele_price_df['time'])
ele_price_df.set_index('time', inplace=True)

In [14]:
simulation_model = EssSimulationModel(es_info)
es_charge_df, es_soc_df, total_load_df = simulation_model.simulation_process(demand_load_df, strategy_df, 0) #4050
origin_balance, opt_balance = simulation_model.revenue_calculation(demand_load_df, es_charge_df, ele_price_df, max_demand_price)

In [15]:
opt_max_demand_load_list, ori_max_demand_load_list = get_monthly_max_load(total_load_df)
opt_max_demand_cost = max_demand_price * sum(opt_max_demand_load_list)
ori_max_demand_cost = max_demand_price * sum(ori_max_demand_load_list)

max_demand_rise_cost = opt_max_demand_cost - ori_max_demand_cost

revenue = origin_balance - opt_balance - max_demand_rise_cost

In [16]:
es_charge_df.to_csv(f"./data/{exp_name}/{node_name}/opt_result/simulation_result_breakcharge_dod95.csv")

In [33]:
revenue

3716452.4269739725

In [34]:
max_demand_rise_cost

1198719.4880000018

In [23]:
3716453.9999945797 + 1198719.4880000018 + 3726389.833086733 + 1188544.4880000018

9830107.809081316

In [22]:
7442843.84

148856.8768

In [25]:
es_charge_df['value'][es_charge_df['value'] > 0].sum() / 4

12351511.018283999

In [35]:
1198719.4880000018 + 1188544.4880000018

2387263.9760000035